# Reflex Colab smoke test (free tier, ~5 min)

Probes a real NVIDIA GPU, syncs the requested GitHub ref, runs smoke tests, captures hardware identity, and publishes a structured result to the `colab-results` branch.

Set the optional Colab Secret `REFLEX_GITHUB_TOKEN` to publish results. Without it, results remain in `/content/reflex_runs` and Drive backup still works.

In [ ]:
# Cell 1 - environment probe.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import platform; print(platform.python_version())"
import pathlib
pathlib.Path('/content/reflex_runs').mkdir(exist_ok=True)
print('probe ok - continue only if a T4/P100 (or better) is listed above')

In [ ]:
# Cell 2 - sync the disposable checkout on every Run all.
# Set REFLEX_REF to a commit SHA to reproduce a specific PR/regression.
import os, subprocess
from pathlib import Path
try:
    from google.colab import userdata
    os.environ['REFLEX_GITHUB_TOKEN'] = userdata.get('REFLEX_GITHUB_TOKEN')
except Exception:
    pass
REPO = Path('/content/reflex')
REFLEX_REF = os.environ.get('REFLEX_REF', 'main')
REMOTE = 'https://github.com/MugiZer/reflex.git'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REMOTE, str(REPO)], check=True)
def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, text=True, capture_output=True).stdout.strip()
git('fetch', 'origin', REFLEX_REF, '--prune')
target = f'origin/{REFLEX_REF}' if REFLEX_REF in ('main', 'reflex-pipeline') else REFLEX_REF
git('checkout', '--detach', target)
COMMIT = git('rev-parse', 'HEAD')
print('using commit:', COMMIT)
print(git('log', '-1', '--oneline'))
%cd /content/reflex

In [ ]:
# Cell 3 - smoke tests. Capture failures so they are published too.
import subprocess
test_command = ['python', '-m', 'pytest', 'tests/test_reflex_ledger.py', 'tests/test_fakegpu.py', 'tests/test_collect.py', '-q']
test_run = subprocess.run(test_command, cwd='/content/reflex', text=True, capture_output=True)
pytest_output = (test_run.stdout + '\n' + test_run.stderr)[-12000:]
print(pytest_output)
print('pytest exit code:', test_run.returncode)

In [ ]:
# Cell 4 - real hardware identity.
from reflex.collect import nvidia_smi_identity
ident = nvidia_smi_identity()
print('identity:', ident)
assert ident['hardware'] != 'unknown', 'no GPU identity - aborting'

In [ ]:
# Cell 5 - publish the structured result to GitHub.
import datetime, uuid
from colab.report_results import publish
result = {
    'run_id': datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8],
    'status': 'passed' if test_run.returncode == 0 else 'failed',
    'commit': COMMIT,
    'requested_ref': REFLEX_REF,
    'hardware': ident.get('hardware'),
    'device': ident.get('device'),
    'driver': ident.get('driver'),
    'cuda': ident.get('cuda'),
    'collector_version': ident.get('collector_version'),
    'pytest_exit_code': test_run.returncode,
    'pytest_output': pytest_output,
}
publish(result)

In [ ]:
# Cell 6 - Drive backup (optional).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil, datetime
    dst = '/content/drive/MyDrive/reflex-colab/%s' % datetime.date.today().isoformat()
    shutil.copytree('/content/reflex_runs', dst, dirs_exist_ok=True)
    print('backed up to', dst)
except Exception as exc:
    print('drive backup skipped:', type(exc).__name__, exc)